<a href="https://colab.research.google.com/github/Jvirado/Entrega-_Final/blob/main/Entrega1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# ============================================================
# CELDA 1. Instalación de dependencias
# ============================================================
!pip -q install pyspark==3.5.1 dask[dataframe,distributed] pyarrow pygeohash pymongo kaggle
!apt-get -qq update && apt-get -qq install -y openjdk-17-jdk-headless > /dev/null

import os
os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-17-openjdk-amd64"
os.environ["SPARK_LOCAL_IP"] = "127.0.0.1"
!pip install --upgrade pyspark pyarrow

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dataproc-spark-connect 1.1.0 requires pyspark[connect]~=4.0.0, but you have pyspark 3.5.1 which is incompatible.
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 455.9/455.9 MB 52.7 MB/s  0:00:06
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 55.0/55.0 MB 78.3 MB/s  0:00:00
  Created wheel for pyspark: filename=pyspark-4.3.0-py2.py3-none-any.whl size=456587423 sha256=20

In [2]:
# ============================================================
# CELDA 2. Credencial Kaggle y descarga del Yelp Dataset
# ============================================================
from google.colab import files
files.upload()

!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/kaggle.json
!chmod 600 ~/.kaggle/kaggle.json

!mkdir -p /content/data
!kaggle datasets download -d yelp-dataset/yelp-dataset -p /content/data

# Descomprimir de forma robusta
import glob, zipfile, tarfile, os

for z in glob.glob("/content/data/*.zip"):
    with zipfile.ZipFile(z, "r") as zf:
        zf.extractall("/content/data")

for t in glob.glob("/content/data/*.tar"):
    with tarfile.open(t, "r") as tf:
        tf.extractall("/content/data/yelp")

print("Archivos encontrados:")
for f in glob.glob("/content/data/**/*.json", recursive=True)[:20]:
    print(f)

Saving kaggle.json to kaggle.json
Dataset URL: https://www.kaggle.com/datasets/yelp-dataset/yelp-dataset
License(s): other
100% 4.07G/4.07G [00:35<00:00, 122MB/s]

Archivos encontrados:
/content/data/yelp_academic_dataset_tip.json
/content/data/yelp_academic_dataset_user.json
/content/data/yelp_academic_dataset_business.json
/content/data/yelp_academic_dataset_review.json
/content/data/yelp_academic_dataset_checkin.json


In [3]:
# ============================================================
# CELDA 3. Configuración del equipo y rutas fijas
# ============================================================
import os

# 1. Elige una ciudad y estado que SÍ existan en el paso anterior
METRO_CITY = "Philadelphia"
METRO_STATE = "PA"
CATEGORY_FAMILY = "Restaurants"
MIN_DIAS_NEGOCIO = 365

# 2. Rutas
BASE = "/content/data"
FILE_BIZ = f"{BASE}/yelp_academic_dataset_business.json"
FILE_REV = f"{BASE}/yelp_academic_dataset_review.json"

# 3. Carpetas
OUT = "/content/lifemap"
os.makedirs(OUT, exist_ok=True)
os.makedirs(f"{OUT}/models", exist_ok=True)
os.makedirs(f"{OUT}/out", exist_ok=True)

print(" Filtros configurados:", METRO_CITY, METRO_STATE, CATEGORY_FAMILY)

 Filtros configurados: Philadelphia PA Restaurants


In [4]:
# ============================================================
# CELDA 4. Filtro de negocios del área/categoría
# ============================================================
import pandas as pd
import dask.dataframe as dd

print("Cargando business.json...")
biz = pd.read_json(FILE_BIZ, lines=True)

# Limpiar la columna categories
biz["categories"] = biz["categories"].fillna("")

# Aplicar los filtros
biz_sel = biz[
    (biz["city"] == METRO_CITY) &
    (biz["state"] == METRO_STATE) &
    (biz["categories"].str.contains(CATEGORY_FAMILY, regex=False, case=False))
].copy()

print(f" Negocios seleccionados: {len(biz_sel)}")


biz_ids = []

if len(biz_sel) > 0:
    print("\nPrimeros 5 negocios:")
    print(biz_sel[["business_id", "name", "city", "categories", "is_open"]].head().to_string())
    biz_ids = biz_sel["business_id"].dropna().unique().tolist()
    print(f"\n Se extrajeron {len(biz_ids)} IDs únicos.")
else:
    print("\n ATENCIÓN: No se encontraron negocios.")
    print("Revisa si escribiste bien la ciudad, el estado o la categoría en la Celda 3.")

Cargando business.json...
 Negocios seleccionados: 5852

Primeros 5 negocios:
               business_id                name          city                                             categories  is_open
3   MTSW4McQd7CbVtyjqoe9mw  St Honore Pastries  Philadelphia  Restaurants, Food, Bubble Tea, Coffee & Tea, Bakeries        1
15  MUTTqe8uqyMdBl186RmNeA            Tuna Bar  Philadelphia                      Sushi Bars, Restaurants, Japanese        1
19  ROeacJQwBeh05Rqg7F6TCg                 BAP  Philadelphia                                    Korean, Restaurants        1
28  QdN72BWoyFypdGJhhI5r7g             Bar One  Philadelphia   Cocktail Bars, Bars, Italian, Nightlife, Restaurants        0
31  Mjboz24M9NlBeiOJKLEd_Q    DeSandro on Main  Philadelphia                        Pizza, Restaurants, Salad, Soup        0

 Se extrajeron 5852 IDs únicos.


In [5]:
# ============================================================
# CELDA 5. Ingesta y limpieza de reseñas con Dask
# ============================================================

if not biz_ids:
    raise ValueError(" ERROR: No hay negocios para procesar.")


print("Iniciando lectura de reseñas con Dask (esto puede tardar un poco)...")
# Leemos con Dask
rev = dd.read_json(FILE_REV, lines=True, blocksize="128MB")

# Seleccionar columnas necesarias
rev = rev[["review_id", "business_id", "stars", "text", "date"]]

# Limpieza básica
rev = rev.dropna(subset=["review_id", "business_id", "stars", "text", "date"])
rev = rev[rev["text"].str.len() > 0]
rev = rev.drop_duplicates(subset=["review_id"])

# Filtrar solo las reseñas de nuestros negocios
rev_sel = rev[rev["business_id"].isin(biz_ids)]

print("Guardando reseñas filtradas en Parquet (por favor espera)...")
# Guardar en formato Parquet
rev_sel.to_parquet(f"{OUT}/reviews_sel.parquet", write_index=False)
print(" Reseñas guardadas exitosamente.")

Iniciando lectura de reseñas con Dask (esto puede tardar un poco)...
Guardando reseñas filtradas en Parquet (por favor espera)...
 Reseñas guardadas exitosamente.


In [6]:
# ============================================================
# CELDA 6  SparkSession con compatibilidad para Parquet
# ============================================================
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("LifeMap-Entrega1")
    .config("spark.driver.memory", "8g")
    .config("spark.sql.shuffle.partitions", "32")
    #  Añade esta línea para leer los timestamps en nanosegundos
    .config("spark.sql.legacy.parquet.nanosAsLong", "true")
    .getOrCreate()
)
print(" Spark iniciado con compatibilidad para Parquet.")

 Spark iniciado con compatibilidad para Parquet.


In [7]:
# ============================================================
# CELDA 7. DataFrames Spark + Vida del Negocio
# ============================================================
from pyspark.sql import functions as F

# 1. Verificamos que la variable biz_sel de la Celda 4 esté en memoria
try:
    biz_sel
except NameError:
    raise ValueError(" ERROR: La variable 'biz_sel' no existe.")

print("Creando DataFrame de Spark para negocios...")
# Seleccionamos las columnas necesarias
biz_cols = biz_sel[[
    "business_id", "name", "city", "state",
    "latitude", "longitude", "stars",
    "review_count", "is_open", "categories"
]].copy()

# Convertimos a DataFrame de Spark
biz_sdf = spark.createDataFrame(biz_cols)

# Extraer la categoría principal (la primera de la lista)
biz_sdf = biz_sdf.withColumn(
    "main_category",
    F.coalesce(F.split(F.col("categories"), ", ").getItem(0), F.lit("Sin categoria"))
)

print("Leyendo reseñas desde el Parquet guardado...")
# 2. Leer el archivo Parquet
rev_sdf = spark.read.parquet(f"{OUT}/reviews_sel.parquet")


rev_sdf = rev_sdf.withColumn(
    "date",
    F.when(
        F.col("date").cast("string").rlike("^[0-9]+$"),
        F.to_date(F.from_unixtime(F.col("date").cast("long") / 1000000000))
    ).otherwise(
        F.to_date("date")
    )
)

# 3. Etiqueta de sentimiento basada en las estrellas
rev_sdf = rev_sdf.withColumn(
    "sentiment",
    F.when(F.col("stars") <= 2, "neg")
     .when(F.col("stars") == 3, "neu")
     .otherwise("pos")
)

print("Calculando la vida de cada negocio...")
# 4. Calcular vida de cada negocio
life = (
    rev_sdf.groupBy("business_id")
    .agg(
        F.min("date").alias("first_review"),
        F.max("date").alias("last_review"),
        F.count("*").alias("n_reviews")
    )
    .withColumn("days_business", F.greatest(F.datediff("last_review", "first_review"), F.lit(1)))
)

# 5. Unir con los datos del negocio (incluyendo latitud, longitud y si está abierto)
life_geo = (
    life.join(
        biz_sdf.select("business_id", "latitude", "longitude", "main_category", "is_open"),
        on="business_id",
        how="left"
    )
    # Si is_open == 0, el evento ocurrió (cerró). Si es 1, está censurado (sigue abierto).
    .withColumn("event", F.when(F.col("is_open") == 0, 1).otherwise(0))
    .withColumn("censored", 1 - F.col("event"))
)

# 6. Guardar en caché para acelerar los cálculos de MapReduce de la Celda 10
life_geo.cache()

print(f"\n Negocios con vida calculada: {life_geo.count()}")
print("Mostrando los primeros 5 registros:")
life_geo.select("business_id", "first_review", "last_review", "days_business", "event", "censored").show(5, truncate=False)

Creando DataFrame de Spark para negocios...
Leyendo reseñas desde el Parquet guardado...
Calculando la vida de cada negocio...

 Negocios con vida calculada: 5852
Mostrando los primeros 5 registros:
+----------------------+------------+-----------+-------------+-----+--------+
|business_id           |first_review|last_review|days_business|event|censored|
+----------------------+------------+-----------+-------------+-----+--------+
|GzWIBhi32It2LMgX2-c_hA|2014-08-27  |2021-11-13 |2635         |0    |1       |
|RhRQhyGHurL9WqRoAn7luQ|2013-09-14  |2019-10-21 |2228         |0    |1       |
|04UD14gamNjLY0IDYVhHJg|2006-12-18  |2020-07-15 |4958         |1    |0       |
|VRnJgj0IvxoksHppE78X2Q|2007-02-02  |2021-12-25 |5440         |0    |1       |
|aCHS5pnDjChkHpFo4-EfKg|2007-02-09  |2022-01-18 |5457         |0    |1       |
+----------------------+------------+-----------+-------------+-----+--------+
only showing top 5 rows


In [8]:
# ============================================================
# CELDA 8. Partición entrenamiento/prueba POR NEGOCIO
# ============================================================
sent = rev_sdf.select("review_id", "business_id", "text", "sentiment", "date").na.drop()

# Asignar fold por negocio (evita que un negocio esté en train y test)
sent = sent.withColumn("fold", F.abs(F.hash("business_id")) % 10)

train = sent.filter("fold < 8").cache()
test = sent.filter("fold >= 8").cache()

print(f"Train: {train.count()} | Test: {test.count()}")
print("\nDistribución train:")
train.groupBy("sentiment").count().show()

Train: 545018 | Test: 142271

Distribución train:
+---------+------+
|sentiment| count|
+---------+------+
|      neu| 71859|
|      neg| 96381|
|      pos|376778|
+---------+------+



In [9]:
# ============================================================
# CELDA 9. Modelo de Sentimiento (Naive Bayes vs Regresión Logística)
# ============================================================
from pyspark.ml import Pipeline
from pyspark.ml.feature import RegexTokenizer, StopWordsRemover, HashingTF, IDF, StringIndexer
from pyspark.ml.classification import NaiveBayes, LogisticRegression
from pyspark.ml.evaluation import MulticlassClassificationEvaluator
from pyspark.mllib.evaluation import MulticlassMetrics

# 1. Definir las etapas del pipeline
label_indexer = StringIndexer(inputCol="sentiment", outputCol="label", handleInvalid="skip")
tokenizer = RegexTokenizer(inputCol="text", outputCol="tokens", pattern="\\W+", toLowercase=True)
remover = StopWordsRemover(inputCol="tokens", outputCol="words")
hashing = HashingTF(inputCol="words", outputCol="tf", numFeatures=2**18)
idf = IDF(inputCol="tf", outputCol="features")

# 2. Definir los algoritmos
nb = NaiveBayes(featuresCol="features", labelCol="label", smoothing=1.0)
lr = LogisticRegression(featuresCol="features", labelCol="label", maxIter=50, regParam=0.01)

# 3. Crear los Pipelines
pipe_nb = Pipeline(stages=[label_indexer, tokenizer, remover, hashing, idf, nb])
pipe_lr = Pipeline(stages=[label_indexer, tokenizer, remover, hashing, idf, lr])

# 4. Función para evaluar y mostrar resultados
def evaluar(model, nombre):
    print(f"Entrenando y evaluando {nombre}...")
    pred = model.transform(test)

    # Calcular F1 Weighted
    f1_eval = MulticlassClassificationEvaluator(
        labelCol="label", predictionCol="prediction", metricName="f1"
    )
    f1w = f1_eval.evaluate(pred)

    # Calcular métricas por clase
    pred_rdd = pred.select("prediction", "label").rdd.map(lambda row: (float(row[0]), float(row[1])))
    metrics = MulticlassMetrics(pred_rdd)

    labels = model.stages[0].labels

    print(f"\n=== Resultados: {nombre} ===")
    print(f"F1 weighted: {f1w:.4f}")
    for i, lab in enumerate(labels):

        print(f"Clase {lab}: Precision={metrics.precision(float(i)):.4f}, "
              f"Recall={metrics.recall(float(i)):.4f}, "
              f"F1={metrics.fMeasure(float(i)):.4f}")

    print("\nMatriz de Confusión:")
    print(metrics.confusionMatrix().toArray())
    print("-" * 40)

    return f1w

# 5. Entrenar Naive Bayes
model_nb = pipe_nb.fit(train)
f1_nb = evaluar(model_nb, "Naive Bayes")

# 6. Entrenar Regresión Logística
model_lr = pipe_lr.fit(train)
f1_lr = evaluar(model_lr, "Regresión Logística")

# 7. Elegir el mejor modelo y guardarlo
best_model = model_nb if f1_nb >= f1_lr else model_lr
nombre_mejor = "Naive Bayes" if f1_nb >= f1_lr else "Regresión Logística"

best_model.write().overwrite().save(f"{OUT}/models/sentiment_v1")
print(f" El mejor modelo fue {nombre_mejor} y se guardó en: {OUT}/models/sentiment_v1")

Entrenando y evaluando Naive Bayes...

=== Resultados: Naive Bayes ===
F1 weighted: 0.7600
Clase pos: Precision=0.9143, Recall=0.7976, F1=0.8520
Clase neg: Precision=0.6847, Recall=0.7080, F1=0.6962
Clase neu: Precision=0.3367, Recall=0.5259, F1=0.4106

Matriz de Confusión:
[[75534.  5049. 14122.]
 [ 1662. 19629.  6432.]
 [ 5418.  3990. 10435.]]
----------------------------------------
Entrenando y evaluando Regresión Logística...

=== Resultados: Regresión Logística ===
F1 weighted: 0.8024
Clase pos: Precision=0.8693, Recall=0.9391, F1=0.9028
Clase neg: Precision=0.7849, Recall=0.7440, F1=0.7639
Clase neu: Precision=0.4613, Recall=0.3180, F1=0.3765

Matriz de Confusión:
[[88937.  1659.  4109.]
 [ 3835. 20627.  3261.]
 [ 9538.  3995.  6310.]]
----------------------------------------
 El mejor modelo fue Regresión Logística y se guardó en: /content/lifemap/models/sentiment_v1


In [10]:
# ============================================================
# CELDA 10. MapReduce de tasa de cierre anualizada
# ============================================================
import pygeohash as pgh
from pyspark.sql.types import StringType

@F.udf(StringType())
def geohash7(lat, lon):
    if lat is None or lon is None: return None
    return pgh.encode(float(lat), float(lon), precision=7)

life_geo = life_geo.withColumn("geohash", geohash7("latitude", "longitude")).cache()

# MapReduce con RDD
rdd = life_geo.select("main_category", "geohash", "event", "days_business").rdd
rdd = rdd.filter(lambda r: r["main_category"] is not None and r["geohash"] is not None)

# (Clave, (cierres, dias_negocio))
rdd_kv = rdd.map(lambda r: ((r["main_category"], r["geohash"]), (int(r["event"]), float(r["days_business"]))))

def sumar(a, b): return (a[0] + b[0], a[1] + b[1])
agg = rdd_kv.reduceByKey(sumar)

# Calcular tasa (Cierres / (dias/365) * 100)
tasa = agg.map(lambda x: (
    x[0][0], x[0][1], x[1][0], x[1][1],
    (x[1][0] / (x[1][1] / 365.0)) * 100.0 if x[1][1] > 0 else None
)).toDF(["main_category", "geohash", "cierres", "dias_negocio", "tasa_cierre_anual"])

tasa = tasa.filter(F.col("dias_negocio") >= MIN_DIAS_NEGOCIO)
print("✅ Tasas de cierre calculadas:")
tasa.show(5, truncate=False)

tasa.write.mode("overwrite").option("header", True).csv(f"{OUT}/out/tasas_cierre")

✅ Tasas de cierre calculadas:
+------------------+-------+-------+------------+-----------------+
|main_category     |geohash|cierres|dias_negocio|tasa_cierre_anual|
+------------------+-------+-------+------------+-----------------+
|Burgers           |dr4e20t|0      |2635.0      |0.0              |
|Breakfast & Brunch|dr4dbne|1      |3726.0      |9.79602791196994 |
|Nightlife         |dr4e1wh|0      |5053.0      |0.0              |
|Nightlife         |dr4e383|1      |7395.0      |4.935767410412441|
|Italian           |dr4e1s2|0      |5047.0      |0.0              |
+------------------+-------+-------+------------+-----------------+
only showing top 5 rows


In [11]:
# ============================================================
# CELDA 11. Balanceo de carga (2, 8, 32, 200 particiones)
# ============================================================
import time, pandas as pd

def medir_particiones(df, n, key=None):
    t0 = time.time()
    d = df.repartition(n, key) if key else df.repartition(n)
    counts = d.rdd.mapPartitionsWithIndex(lambda i, it: [(i, sum(1 for _ in it))]).collect()
    dt = time.time() - t0
    vals = [c[1] for c in counts]
    return dt, max(vals), min(vals)

resultados = []
for n in [2, 8, 32, 200]:
    for key in [None, "main_category"]:
        for rep in range(3):
            dt, maxp, minp = medir_particiones(life_geo, n, key)
            resultados.append({
                "particiones": n, "reparto": "con_clave" if key else "sin_clave",
                "repeticion": rep + 1, "segundos": dt, "max_filas_particion": maxp
            })

balance_df = pd.DataFrame(resultados)
print(" Tabla de tiempos generada.")
display(balance_df.groupby(["particiones", "reparto"])["segundos"].agg(["mean", "std"]).reset_index())

 Tabla de tiempos generada.


,particiones,reparto,mean,std
0,2,con_clave,0.688950,0.090758
1,2,sin_clave,0.835536,0.188792
2,8,con_clave,1.135864,0.181679
3,8,sin_clave,1.092270,0.104055
4,32,con_clave,1.706054,0.079621
5,32,sin_clave,1.875923,0.014131
6,200,con_clave,6.556662,0.084847
7,200,sin_clave,7.687673,0.724766


In [12]:
# ============================================================
# CELDA 12. Exportar muestra GeoJSON
# ============================================================
import json

muestra = life_geo.select(
    "business_id", "main_category", "latitude", "longitude", "days_business", "event", "censored"
).limit(2000).toPandas()

docs = muestra.to_dict("records")
for d in docs:
    d["location"] = {"type": "Point", "coordinates": [float(d["longitude"]), float(d["latitude"])]}

with open(f"{OUT}/out/businesses_geojson_sample.json", "w") as f:
    for d in docs:
        f.write(json.dumps(d) + "\n")

print(f" Muestra GeoJSON guardada con {len(docs)} documentos.")

✅ Muestra GeoJSON guardada con 2000 documentos.
